# SPATIAL INTELLIGENCE - PART 1

In [135]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [136]:
from collections import defaultdict, deque

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph

## 2. Check the TopologicPy Version

In [137]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.19) is OLDER than the latest version (0.9.20) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [138]:
renderer = "vscode"

## 4. Import the OBJ file with rooms and doors

In [139]:
obj_path = r"C:\Users\Arq. David Agudelo\GRAPHML_DAVID_AGUDELO_2026\Test_04.obj"
objects = Topology.ByOBJPath(obj_path, selfMerge=True)
model = objects[0]

print("Imported topologies:", len(objects))
print("Imported type:", Topology.TypeAsString(model))
print("Detected cells in imported model:", len(Topology.Cells(model)))

Imported topologies: 1
Imported type: Cluster
Detected cells in imported model: 58


## 5. Build the spatial model from the imported geometry

In [140]:
all_cells = Topology.Cells(model)
cell_data = sorted([(Cell.Volume(c), i, c) for i, c in enumerate(all_cells)], reverse=True)

room_count = 10
room_ids = [item[1] for item in cell_data[:room_count]]
room_cells = [item[2] for item in cell_data[:room_count]]
door_cells = [item[2] for item in cell_data[room_count:]]

house = CellComplex.ByCells(room_cells)

print("Room cells kept:", len(room_cells))
print("Door connector cells found:", len(door_cells))

Topology.Show(house,
              faceColor=[210,210,250],
              faceOpacity=0.6,
              edgeColor="white",
              edgeWidth=2,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Room cells kept: 10
Door connector cells found: 48


## 5b. Compute and print the room centroids
Calculate the centroid of each room cell and sort them by spatial position (Y, then X).

In [163]:
room_centers = [(i, Topology.Centroid(c)) for i, c in zip(room_ids, room_cells)]
room_centers = sorted(room_centers, key=lambda item: (Vertex.Y(item[1]), Vertex.X(item[1])))

for n, (cell_id, center) in enumerate(room_centers, start=1):
    print(f"Room {n}: X={Vertex.X(center):.2f}, Y={Vertex.Y(center):.2f}, Z={Vertex.Z(center):.2f}")

Room 1: X=78.03, Y=19.17, Z=4.92
Room 2: X=41.96, Y=23.10, Z=4.74
Room 3: X=71.37, Y=44.57, Z=5.01
Room 4: X=92.82, Y=47.48, Z=5.57
Room 5: X=40.19, Y=80.42, Z=5.04
Room 6: X=72.91, Y=84.00, Z=5.15
Room 7: X=98.40, Y=93.15, Z=5.14
Room 8: X=73.33, Y=117.86, Z=5.11
Room 9: X=38.82, Y=126.74, Z=5.14
Room 10: X=64.22, Y=147.19, Z=5.14


## 6. Compute the room centroids
Create a vertex at the centroid of each of the `room_count` rooms.

In [166]:
center_vertices = []
for cell_id, center in room_centers:
    v = Vertex.ByCoordinates(Vertex.X(center), Vertex.Y(center), Vertex.Z(center))
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    Topology.SetDictionary(v, d)
    center_vertices.append(v)

g0 = Graph.ByVerticesEdges(center_vertices, [])
print(f"Room center vertices (room_count={room_count}):", len(Graph.Vertices(g0)))

Room center vertices (room_count=10): 10


## 6b. Show the room centers
Visualize only the centroid vertices of each room on top of the geometry, before adding any edges.

In [167]:
Topology.Show(house, g0,
              vertexSizeKey="size",
              vertexColorKey="color",
              faceOpacity=0.3,
              width=500,
              height=500,
              backgroundColor="white",
              renderer=renderer)

## 7. Derive adjacency graph

In [168]:
room_vertex_map = {}
for cell_id, center in room_centers:
    v = Vertex.ByCoordinates(Vertex.X(center), Vertex.Y(center), Vertex.Z(center))
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    Topology.SetDictionary(v, d)
    room_vertex_map[cell_id] = v

adjacency_edges = []
for i in range(len(room_ids)):
    for j in range(i+1, len(room_ids)):
        shared_faces = Topology.SharedFaces(room_cells[i], room_cells[j])
        if shared_faces:
            e = Edge.ByVertices([room_vertex_map[room_ids[i]], room_vertex_map[room_ids[j]]])
            d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
            Topology.SetDictionary(e, d)
            adjacency_edges.append(e)

g1 = Graph.ByVerticesEdges(list(room_vertex_map.values()), adjacency_edges)
print("Adjacency graph vertices:", len(Graph.Vertices(g1)))
print("Adjacency graph edges:", len(Graph.Edges(g1)))

Adjacency graph vertices: 10
Adjacency graph edges: 17


## 8. Assign visual attributes to the vertices and edges of the graph

In [143]:
print("Visual attributes are passed directly to Topology.Show to avoid node displacement.")

Visual attributes are passed directly to Topology.Show to avoid node displacement.


## 9. Show the geometry and the graph

In [169]:
Topology.Show(house, g1,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              width=500,
              height=500,
              backgroundColor="white",
              renderer=renderer
              )

## 10. Derive the access graph through the door connectors

In [170]:
adjacency = defaultdict(set)
for i, cell in enumerate(all_cells):
    neighbors = Topology.AdjacentTopologies(cell, model, topologyType="cell") or []
    for neighbor in neighbors:
        for j, other in enumerate(all_cells):
            if Topology.IsSame(neighbor, other):
                adjacency[i].add(j)
                break

room_set = set(room_ids)
small_set = set(range(len(all_cells))) - room_set
room_vertex_map = {}
for cell_id, center in room_centers:
    v = Vertex.ByCoordinates(Vertex.X(center), Vertex.Y(center), Vertex.Z(center))
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    Topology.SetDictionary(v, d)
    room_vertex_map[cell_id] = v

access_pairs = set()
for room_id in room_ids:
    queue = deque([room_id])
    visited = {room_id}
    while queue:
        current = queue.popleft()
        for nb in adjacency[current]:
            if nb in visited:
                continue
            visited.add(nb)
            if nb in small_set:
                queue.append(nb)
            elif nb in room_set and nb != room_id:
                access_pairs.add(tuple(sorted((room_id, nb))))

access_edges = []
for a, b in sorted(access_pairs):
    e = Edge.ByVertices([room_vertex_map[a], room_vertex_map[b]])
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    Topology.SetDictionary(e, d)
    access_edges.append(e)

g2 = Graph.ByVerticesEdges(list(room_vertex_map.values()), access_edges)

print("Access graph vertices:", len(Graph.Vertices(g2)))
print("Access graph edges:", len(Graph.Edges(g2)))

Access graph vertices: 10
Access graph edges: 19


## 11. Assign visual attributes to the vertices and edges of the graph

In [146]:
print("Visual attributes are passed directly to Topology.Show to avoid node displacement.")

Visual attributes are passed directly to Topology.Show to avoid node displacement.


## 13. Identify the doors
The imported OBJ represents doors as small connector cells. We group connected small cells into one door component and detect which rooms each one touches.

In [148]:
door_components = []
seen = set()

room_centers = [(i, Topology.Centroid(c)) for i, c in zip(room_ids, room_cells)]
room_centers = sorted(room_centers, key=lambda item: (Vertex.Y(item[1]), Vertex.X(item[1])))
room_label = {cell_id: f"Room {n}" for n, (cell_id, _) in enumerate(room_centers, start=1)}
room_vertex_map = {cell_id: center for cell_id, center in room_centers}

for s in sorted(small_set):
    if s in seen:
        continue
    queue = deque([s])
    seen.add(s)
    component = []
    while queue:
        current = queue.popleft()
        component.append(current)
        for nb in adjacency[current]:
            if nb in small_set and nb not in seen:
                seen.add(nb)
                queue.append(nb)

    touched_rooms = sorted({nb for cid in component for nb in adjacency[cid] if nb in room_set})
    if not touched_rooms:
        continue

    centers = [Topology.Centroid(all_cells[cid]) for cid in component]
    x = sum(Vertex.X(v) for v in centers)/len(centers)
    y = sum(Vertex.Y(v) for v in centers)/len(centers)
    z = sum(Vertex.Z(v) for v in centers)/len(centers)
    door_center = Vertex.ByCoordinates(x, y, z)
    door_components.append({"cells": component, "rooms": touched_rooms, "center": door_center})

print("Identified door groups:", len(door_components))
for i, door in enumerate(door_components, start=1):
    room_names = [room_label[r] for r in door["rooms"]]
    print(f"Door {i}: connects {room_names}")

Identified door groups: 21
Door 1: connects ['Room 5', 'Room 2']
Door 2: connects ['Room 5', 'Room 3']
Door 3: connects ['Room 5', 'Room 6']
Door 4: connects ['Room 5', 'Room 6', 'Room 9', 'Room 8']
Door 5: connects ['Room 5', 'Room 9']
Door 6: connects ['Room 2']
Door 7: connects ['Room 2']
Door 8: connects ['Room 2', 'Room 1']
Door 9: connects ['Room 2', 'Room 3']
Door 10: connects ['Room 3', 'Room 6']
Door 11: connects ['Room 3', 'Room 1']
Door 12: connects ['Room 3', 'Room 1', 'Room 4']
Door 13: connects ['Room 3', 'Room 4']
Door 14: connects ['Room 3', 'Room 6', 'Room 4']
Door 15: connects ['Room 6', 'Room 8']
Door 16: connects ['Room 6', 'Room 7']
Door 17: connects ['Room 9', 'Room 8']
Door 18: connects ['Room 9', 'Room 10']
Door 19: connects ['Room 1']
Door 20: connects ['Room 8', 'Room 10']
Door 21: connects ['Room 8', 'Room 7']


## 13b. Print the door centers

In [173]:
print(f"{'Door':<10} {'X':>8} {'Y':>8} {'Z':>8}   Connects")
print("-" * 60)
for i, door in enumerate(door_components, start=1):
    dc = door["center"]
    room_names = [room_label[r] for r in door["rooms"]]
    print(f"Door {i:<5} {Vertex.X(dc):8.2f} {Vertex.Y(dc):8.2f} {Vertex.Z(dc):8.2f}   {', '.join(room_names)}")

Door              X        Y        Z   Connects
------------------------------------------------------------
Door 1        28.00    53.00     3.69   Room 5, Room 2
Door 2        56.00    58.00     4.00   Room 5, Room 3
Door 3        56.00    84.50     3.69   Room 5, Room 6
Door 4        53.45   106.00     4.00   Room 5, Room 6, Room 9, Room 8
Door 5        25.63   106.00     3.43   Room 5, Room 9
Door 6        28.00    -0.00     3.69   Room 2
Door 7        56.00     3.29     4.00   Room 2
Door 8        56.00    15.79     3.69   Room 2, Room 1
Door 9        56.00    39.00     4.00   Room 2, Room 3
Door 10       71.07    63.00     3.43   Room 3, Room 6
Door 11       71.00    25.00     3.69   Room 3, Room 1
Door 12       86.47    27.69     4.00   Room 3, Room 1, Room 4
Door 13       86.00    46.32     3.43   Room 3, Room 4
Door 14       87.00    63.00     4.00   Room 3, Room 6, Room 4
Door 15       72.00   106.00     3.69   Room 6, Room 8
Door 16       88.00    88.50     3.69   Room 6, R

## 13c. Show the geometry and the centers of doors

In [ ]:
door_center_vertices = []
for i, door in enumerate(door_components, start=1):
    dc = door["center"]
    v = Vertex.ByCoordinates(Vertex.X(dc), Vertex.Y(dc), Vertex.Z(dc))
    d = Dictionary.ByKeysValues(["size", "color"], [14, "blue"])
    Topology.SetDictionary(v, d)
    door_center_vertices.append(v)

g_doors = Graph.ByVerticesEdges(door_center_vertices, [])

Topology.Show(house, g_doors,
              vertexSizeKey="size",
              vertexColorKey="color",
              faceOpacity=0.3,
              width=500,
              height=500,
              backgroundColor="white",
              renderer=renderer)

## 14. Generate the door-space connection graph
This graph is bipartite: one type of node is a room, and the other type is a door.

In [172]:
room_graph_vertices = []
room_vertex_map_g3 = {}
for cell_id, center in room_centers:
    v = Vertex.ByCoordinates(Vertex.X(center), Vertex.Y(center), Vertex.Z(center))
    d = Dictionary.ByKeysValues(["label", "color", "size"], [room_label[cell_id], "red", 18])
    Topology.SetDictionary(v, d)
    room_graph_vertices.append(v)
    room_vertex_map_g3[cell_id] = v

door_graph_vertices = []
door_graph_edges = []

for i, door in enumerate(door_components, start=1):
    door_lbl = f"Door {i}"
    dc = door["center"]
    dv = Vertex.ByCoordinates(Vertex.X(dc), Vertex.Y(dc), Vertex.Z(dc))
    Topology.SetDictionary(dv, Dictionary.ByKeysValues(["label", "color", "size"], [door_lbl, "blue", 12]))
    door_graph_vertices.append(dv)

    for room_id in door["rooms"]:
        e = Edge.ByVertices([dv, room_vertex_map_g3[room_id]])
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [2, "gray"]))
        door_graph_edges.append(e)

g3 = Graph.ByVerticesEdges(room_graph_vertices + door_graph_vertices, door_graph_edges)

print("Door-space graph vertices:", len(Graph.Vertices(g3)))
print("Door-space graph edges:", len(Graph.Edges(g3)))

Topology.Show(house, g3,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              vertexLabelKey="label",
              showVertexLabel=True,
              faceOpacity=0.15,
              width=900,
              height=650,
              backgroundColor="white",
              renderer=renderer)

Door-space graph vertices: 31
Door-space graph edges: 43
